# ChpWiseDB RE-RANKER — Clean Pipeline
## Sections: Imports → PDF → Chunks → Gold Standard → Embeddings → Retrieval → Rerankers → Metrics → Evaluation → Export

## 1. Imports & Config

In [6]:
import re
import time
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from sentence_transformers import CrossEncoder

# ── Paths & model choices ──────────────────────────────────────────────────
JCI_PDF       = "/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf"
CROSS_MODEL   = "cross-encoder/ms-marco-MiniLM-L-6-v2"
QWEN_MODEL_ID = "Qwen/Qwen3-Reranker-4B"

# ── Embedding models to evaluate ──────────────────────────────────────────
EMBEDDING_MODELS = {
    "MiniLM":    "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small": "BAAI/bge-small-en-v1.5",
    "BGE-large": "BAAI/bge-large-en-v1.5",
}

# ── Retrieval settings ─────────────────────────────────────────────────────
PER_DB_K     = 8
FINAL_K      = 10
QWEN_INPUT_K = 10

# ── Evaluation settings ───────────────────────────────────────────────────
HIT_AT_K = 5
K_VALUES = [1, 3, 5]

# ── JCI chapter codes ─────────────────────────────────────────────────────
VALID_CODES = {
    "APR", "IPSG", "ACC", "PCC", "AOP", "COP", "ASC", "MMU",
    "QPS", "PCI", "GLD", "FMS", "SQE", "MOI", "MPE", "HRP",
}
END_OF_BOOK_MARKERS = ["glossary", "index", "appendix", "references"]

print("Config loaded.")

Config loaded.


## 2. Load PDF

In [7]:
def load_pdf(path: str):
    loader = PyPDFLoader(path)
    pages  = loader.load()
    print(f"Loaded {len(pages)} pages.")
    return pages

pages = load_pdf(JCI_PDF)

Loaded 406 pages.


## 3. Group Pages by Chapter & Build Chunks

In [8]:
def detect_chapter(text: str):
    for line in text.split("\n"):
        m = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line.strip())
        if m and m.group(1) in VALID_CODES:
            return m.group(1)
    return None


def is_end_of_book(text: str) -> bool:
    return any(marker in text.strip()[:200].lower() for marker in END_OF_BOOK_MARKERS)


def group_pages_by_chapter(pages) -> dict:
    chapter_pages = defaultdict(list)
    current = "UNKNOWN"
    for page in pages:
        detected = detect_chapter(page.page_content)
        if detected:
            current = detected
        chapter_pages[current].append(page)

    if "HRP" in chapter_pages:
        before = len(chapter_pages["HRP"])
        chapter_pages["HRP"] = [
            p for p in chapter_pages["HRP"] if not is_end_of_book(p.page_content)
        ]
        print(f"HRP: removed {before - len(chapter_pages['HRP'])} end-of-book pages.")

    print("\n=== Chapter page counts ===")
    for ch, pgs in chapter_pages.items():
        print(f"  {ch}: {len(pgs)} pages")
    return chapter_pages


chapter_pages = group_pages_by_chapter(pages)

HRP: removed 13 end-of-book pages.

=== Chapter page counts ===
  UNKNOWN: 2 pages
  FMS: 42 pages
  APR: 18 pages
  ACC: 22 pages
  PCC: 21 pages
  AOP: 30 pages
  COP: 32 pages
  ASC: 16 pages
  QPS: 15 pages
  PCI: 27 pages
  GLD: 32 pages
  SQE: 28 pages
  MOI: 25 pages
  MPE: 13 pages
  HRP: 32 pages
  IPSG: 16 pages
  MMU: 22 pages


In [9]:
def build_chapter_chunks(chapter_pages: dict, chunk_size=1000, chunk_overlap=250) -> dict:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "],
    )
    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        full_text = "\n".join(p.page_content for p in pg_list)
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text)
        full_text = re.sub(r'\n{3,}', '\n\n', full_text)

        toc = re.search(r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements',
                        full_text, re.IGNORECASE)
        if toc:
            full_text = full_text[toc.start():]

        pattern = rf'(?:\n|^)(?:Standard|Requirement)\s+({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?(?:\s*\.\s*\d+)?)\b'
        splits  = re.split(pattern, full_text)

        docs = []
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={"chapter": chapter, "standard_code": "INTRO"},
            ))

        for i in range(1, len(splits), 2):
            if i + 1 >= len(splits):
                break
            code    = splits[i].strip().replace(" ", "")
            content = f"{code}\n{splits[i + 1]}"

            if len(content.strip()) < 100:
                continue
            if len(content) <= chunk_size:
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={"chapter": chapter, "standard_code": code},
                ))
            else:
                for j, sub in enumerate(splitter.split_text(content)):
                    if len(sub.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=sub.strip(),
                        metadata={"chapter": chapter, "standard_code": code, "chunk_index": j},
                    ))

        chapter_chunks[chapter] = docs
        unique = len(set(d.metadata["standard_code"] for d in docs))
        print(f"  {chapter}: {len(docs)} chunks across {unique} standards")

    return chapter_chunks


chapter_chunks = build_chapter_chunks(chapter_pages)

  UNKNOWN: 1 chunks across 1 standards
  FMS: 131 chunks across 27 standards
  APR: 32 chunks across 12 standards
  ACC: 80 chunks across 19 standards
  PCC: 79 chunks across 22 standards
  AOP: 114 chunks across 36 standards
  COP: 129 chunks across 29 standards
  ASC: 50 chunks across 16 standards
  QPS: 56 chunks across 12 standards
  PCI: 106 chunks across 22 standards
  GLD: 142 chunks across 34 standards
  SQE: 128 chunks across 26 standards
  MOI: 95 chunks across 17 standards
  MPE: 23 chunks across 8 standards
  HRP: 140 chunks across 10 standards
  IPSG: 76 chunks across 14 standards
  MMU: 85 chunks across 19 standards


## 4. Gold Standard Queries

In [10]:
gold_standard = [
    # ACC.1 — Screening and Admission
    {"query": "What screening criteria should be used when a patient arrives at the emergency department?", "expected_standard": "ACC.1",     "expected_chapter": "ACC"},
    {"query": "How should hospitals handle patients with communicable diseases during screening?",           "expected_standard": "ACC.1",     "expected_chapter": "ACC"},
    # ACC.1.1 — Delays in Care
    {"query": "How should hospitals inform patients about delays in diagnostic or treatment services?",      "expected_standard": "ACC.1.1",   "expected_chapter": "ACC"},
    # ACC.2 — Patient Flow
    {"query": "How should hospitals manage patient flow to prevent emergency department overcrowding?",      "expected_standard": "ACC.2",     "expected_chapter": "ACC"},
    {"query": "What are the requirements for managing patients waiting for inpatient beds in the ED?",       "expected_standard": "ACC.2",     "expected_chapter": "ACC"},
    # ACC.2.2 — Admission Information
    {"query": "What information must be provided to patients when they are admitted to the hospital?",       "expected_standard": "ACC.2.2",   "expected_chapter": "ACC"},
    # ACC.2.3 — ICU / Specialised Services
    {"query": "What criteria should be established for admission to intensive care units?",                  "expected_standard": "ACC.2.3",   "expected_chapter": "ACC"},
    {"query": "How should hospitals determine when patients no longer need intensive care?",                 "expected_standard": "ACC.2.3",   "expected_chapter": "ACC"},
    {"query": "What are the requirements for psychiatric patient admission criteria?",                       "expected_standard": "ACC.2.3",   "expected_chapter": "ACC"},
    {"query": "What criteria should hospitals use for admission and discharge from intensive or specialized care departments?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    # ACC.3.1 — Continuity of Care
    {"query": "Who should be responsible for coordinating a patient's care throughout their hospital stay?", "expected_standard": "ACC.3.1",  "expected_chapter": "ACC"},
    {"query": "What processes should be in place to ensure continuity when care responsibility is transferred?", "expected_standard": "ACC.3.1", "expected_chapter": "ACC"},
    # ACC.4 — Discharge Planning
    {"query": "When should discharge planning begin for hospitalized patients?",                            "expected_standard": "ACC.4",     "expected_chapter": "ACC"},
    # ACC.4.1 — Discharge Education
    {"query": "What education must be provided to patients and families before discharge?",                  "expected_standard": "ACC.4.1",   "expected_chapter": "ACC"},
    {"query": "What education and follow-up is required for a complex care patient being discharged home?",  "expected_standard": "ACC.4.1",   "expected_chapter": "ACC"},
    # ACC.4.2 — Discharge Summary
    {"query": "What information must be included in a patient's discharge summary?",                        "expected_standard": "ACC.4.2",   "expected_chapter": "ACC"},
    # ACC.4.2.1 — ED Documentation
    {"query": "What specific information must be documented for emergency department patients?",             "expected_standard": "ACC.4.2.1", "expected_chapter": "ACC"},
    # ACC.4.3 — Complex Care
    {"query": "How should hospitals manage information for patients with multiple complex diagnoses?",       "expected_standard": "ACC.4.3",   "expected_chapter": "ACC"},
    # ACC.4.4 — Against Medical Advice
    {"query": "What procedures should be followed when a patient wants to leave against medical advice?",   "expected_standard": "ACC.4.4",   "expected_chapter": "ACC"},
    {"query": "What documentation is required when a patient leaves without notifying staff?",              "expected_standard": "ACC.4.4.1", "expected_chapter": "ACC"},
    # ACC.5 — Transfer
    {"query": "What criteria should be used to determine if a patient needs to be transferred to another facility?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
    # ACC.5.1 — Transfer Documentation
    {"query": "What information must be provided to the receiving hospital when transferring a patient?",    "expected_standard": "ACC.5.1",   "expected_chapter": "ACC"},
    {"query": "What are all the documentation requirements when a patient is transferred from the ICU?",     "expected_standard": "ACC.5.1",   "expected_chapter": "ACC"},
    # ACC.6 — Transportation
    {"query": "What are the hospital's responsibilities for patient transportation services?",               "expected_standard": "ACC.6",     "expected_chapter": "ACC"},
    {"query": "What infection control measures should be considered during patient transportation?",         "expected_standard": "ACC.6",     "expected_chapter": "ACC"},
]

print(f"Gold standard: {len(gold_standard)} queries")
for std, n in sorted(Counter(q["expected_standard"] for q in gold_standard).items()):
    print(f"  {std}: {n}")

Gold standard: 25 queries
  ACC.1: 2
  ACC.1.1: 1
  ACC.2: 2
  ACC.2.2: 1
  ACC.2.3: 4
  ACC.3.1: 2
  ACC.4: 1
  ACC.4.1: 2
  ACC.4.2: 1
  ACC.4.2.1: 1
  ACC.4.3: 1
  ACC.4.4: 1
  ACC.4.4.1: 1
  ACC.5: 1
  ACC.5.1: 2
  ACC.6: 2


## 5. Build & Save FAISS Databases
> Run once. Skips models that already have saved DBs.

In [11]:
import os

DB_SAVE_DIR = "saved_dbs"
os.makedirs(DB_SAVE_DIR, exist_ok=True)

for model_short_name, model_string in EMBEDDING_MODELS.items():

    print(f"\n{'='*60}")
    print(f"  Building DBs for: {model_short_name}")
    print(f"{'='*60}\n")

    model_dir = os.path.join(DB_SAVE_DIR, model_short_name)
    if os.path.exists(model_dir):
        print(f"  DBs already exist at {model_dir} — skipping.")
        continue

    os.makedirs(model_dir, exist_ok=True)

    embeddings = HuggingFaceEmbeddings(
        model_name=model_string,
        encode_kwargs={"normalize_embeddings": True},
    )

    for chapter, docs in chapter_chunks.items():
        if docs:
            db = FAISS.from_documents(docs, embeddings)
            chapter_path = os.path.join(model_dir, chapter)
            db.save_local(chapter_path)
            print(f"    Saved: {chapter_path}")

    print(f"\n  [{model_short_name}] All DBs saved.")

print(f"\n{'='*60}")
print(f"  ALL DATABASES BUILT AND SAVED")
print(f"{'='*60}")


  Building DBs for: MiniLM



/var/folders/r6/6d8w4h7s47x81h2h03y77ys40000gn/T/ipykernel_45136/1989939288.py:19: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 974.14it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Saved: saved_dbs/MiniLM/UNKNOWN
    Saved: saved_dbs/MiniLM/FMS
    Saved: saved_dbs/MiniLM/APR
    Saved: saved_dbs/MiniLM/ACC
    Saved: saved_dbs/MiniLM/PCC
    Saved: saved_dbs/MiniLM/AOP
    Saved: saved_dbs/MiniLM/COP
    Saved: saved_dbs/MiniLM/ASC
    Saved: saved_dbs/MiniLM/QPS
    Saved: saved_dbs/MiniLM/PCI
    Saved: saved_dbs/MiniLM/GLD
    Saved: saved_dbs/MiniLM/SQE
    Saved: saved_dbs/MiniLM/MOI
    Saved: saved_dbs/MiniLM/MPE
    Saved: saved_dbs/MiniLM/HRP
    Saved: saved_dbs/MiniLM/IPSG
    Saved: saved_dbs/MiniLM/MMU

  [MiniLM] All DBs saved.

  Building DBs for: BGE-small



Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1348.49it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Saved: saved_dbs/BGE-small/UNKNOWN
    Saved: saved_dbs/BGE-small/FMS
    Saved: saved_dbs/BGE-small/APR
    Saved: saved_dbs/BGE-small/ACC
    Saved: saved_dbs/BGE-small/PCC
    Saved: saved_dbs/BGE-small/AOP
    Saved: saved_dbs/BGE-small/COP
    Saved: saved_dbs/BGE-small/ASC
    Saved: saved_dbs/BGE-small/QPS
    Saved: saved_dbs/BGE-small/PCI
    Saved: saved_dbs/BGE-small/GLD
    Saved: saved_dbs/BGE-small/SQE
    Saved: saved_dbs/BGE-small/MOI
    Saved: saved_dbs/BGE-small/MPE
    Saved: saved_dbs/BGE-small/HRP
    Saved: saved_dbs/BGE-small/IPSG
    Saved: saved_dbs/BGE-small/MMU

  [BGE-small] All DBs saved.

  Building DBs for: BGE-large



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1134.38it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    Saved: saved_dbs/BGE-large/UNKNOWN
    Saved: saved_dbs/BGE-large/FMS
    Saved: saved_dbs/BGE-large/APR
    Saved: saved_dbs/BGE-large/ACC
    Saved: saved_dbs/BGE-large/PCC
    Saved: saved_dbs/BGE-large/AOP
    Saved: saved_dbs/BGE-large/COP
    Saved: saved_dbs/BGE-large/ASC
    Saved: saved_dbs/BGE-large/QPS
    Saved: saved_dbs/BGE-large/PCI
    Saved: saved_dbs/BGE-large/GLD
    Saved: saved_dbs/BGE-large/SQE
    Saved: saved_dbs/BGE-large/MOI
    Saved: saved_dbs/BGE-large/MPE
    Saved: saved_dbs/BGE-large/HRP
    Saved: saved_dbs/BGE-large/IPSG
    Saved: saved_dbs/BGE-large/MMU

  [BGE-large] All DBs saved.

  ALL DATABASES BUILT AND SAVED


## 6. Retrieval

In [12]:
def retrieve(query: str, dbs: dict, per_db_k: int = PER_DB_K, final_k: int = FINAL_K):
    """
    Query every chapter DB, globally sort by FAISS L2 distance
    (lower = better), deduplicate, return top final_k docs.
    Returns list of (doc, raw_l2_score).
    """
    pool = []
    for chapter, db in dbs.items():
        for doc, score in db.similarity_search_with_score(query, k=per_db_k):
            doc.metadata["chapter"] = chapter
            pool.append((doc, score))

    pool.sort(key=lambda x: x[1])

    seen, unique = set(), []
    for doc, score in pool:
        key = (doc.page_content, doc.metadata.get("standard_code"))
        if key not in seen:
            unique.append((doc, score))
            seen.add(key)

    return unique[:final_k]

## 7. Score Normalisation & Rerankers

All three rerankers output different raw score scales. We normalise everything to 0–1 (higher = more relevant) so scores are comparable across rerankers.

- **Baseline (FAISS L2):** Inverted then min-max scaled per query
- **CrossEncoder:** Sigmoid applied to raw logits
- **Qwen:** Already a 0–1 probability — no change needed

In [13]:
def normalise_scores(scores: list, mode: str) -> list:
    """
    Convert raw reranker scores to 0-1 probability scale.

    baseline (FAISS L2 distance):
        Lower = more similar. Invert then min-max scale per query.
        Result: 1.0 = most similar chunk, 0.0 = least similar.

    cross (CrossEncoder logit):
        Unbounded range, typically -11 to +9 in practice.
        Sigmoid: score = 1 / (1 + exp(-x))
        Standard way to convert logits to probabilities.
        Monotonic so rank ordering is preserved exactly.

    qwen (log_softmax + exp probability):
        Already 0-1. No transformation needed.
    """
    arr = np.array(scores, dtype=float)

    if mode == "baseline":
        arr = -arr                          # invert: lower L2 → higher relevance
        mn, mx = arr.min(), arr.max()
        if mx > mn:
            arr = (arr - mn) / (mx - mn)   # scale to 0-1
        else:
            arr = np.ones_like(arr)         # all identical scores → all 1.0

    elif mode == "cross":
        arr = 1.0 / (1.0 + np.exp(-arr))   # sigmoid

    # qwen: return as-is
    return [round(float(x), 4) for x in arr]


# ── CrossEncoder ──────────────────────────────────────────────────────────
def rerank_cross(query: str, retrieved: list, model) -> list:
    """
    Returns list of (doc, raw_score, norm_score) sorted by raw score descending.
    raw_score  = CrossEncoder logit (unbounded)
    norm_score = sigmoid(logit) → 0-1
    """
    docs     = [doc for doc, _ in retrieved]
    pairs    = [(query, doc.page_content) for doc in docs]
    raw      = model.predict(pairs).tolist()
    normed   = normalise_scores(raw, "cross")
    combined = sorted(zip(docs, raw, normed), key=lambda x: x[1], reverse=True)
    return list(combined)


cross_model = CrossEncoder(CROSS_MODEL)
print("CrossEncoder loaded.")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1653.43it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CrossEncoder loaded.


In [14]:
def load_qwen():
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"Loading Qwen: {QWEN_MODEL_ID}")

    tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, padding_side='left')
    model     = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL_ID, device_map="auto"
    ).eval()

    token_true_id  = tokenizer.convert_tokens_to_ids("yes")
    token_false_id = tokenizer.convert_tokens_to_ids("no")
    max_length     = 8192

    prefix = (
        "<|im_start|>system\n"
        "Judge whether the Document meets the requirements based on the "
        "Query and the Instruct provided. "
        "Note that the answer can only be \"yes\" or \"no\"."
        "<|im_end|>\n<|im_start|>user\n"
    )
    suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

    prefix_tokens = tokenizer.encode(prefix, add_special_tokens=False)
    suffix_tokens = tokenizer.encode(suffix, add_special_tokens=False)

    return (tokenizer, model, token_true_id, token_false_id,
            max_length, prefix_tokens, suffix_tokens)


def rerank_qwen(query, retrieved, tokenizer, model,
                token_true_id, token_false_id,
                max_length, prefix_tokens, suffix_tokens,
                expected="UNKNOWN"):
    """
    Returns list of (doc, raw_score, norm_score) sorted by score descending.
    raw_score == norm_score for Qwen (already 0-1 probability).
    """
    import torch

    instruction = (
        "Given a hospital accreditation policy query, retrieve the document "
        "chunk that contains the relevant standard, intent, or measurable "
        "elements that answer the query."
    )

    def format_input(q, doc):
        return (
            f"<Instruct>: {instruction}\n"
            f"<Query>: {q}\n"
            f"<Document>: {doc}"
        )

    def process_inputs(pairs):
        inputs = tokenizer(
            pairs,
            padding=False,
            truncation='longest_first',
            return_attention_mask=False,
            max_length=max_length - len(prefix_tokens) - len(suffix_tokens)
        )
        for i, ele in enumerate(inputs['input_ids']):
            inputs['input_ids'][i] = prefix_tokens + ele + suffix_tokens
        inputs = tokenizer.pad(
            inputs, padding=True, return_tensors="pt", max_length=max_length
        )
        for key in inputs:
            inputs[key] = inputs[key].to(model.device)
        return inputs

    def compute_logits(inputs):
        batch_scores = model(**inputs).logits[:, -1, :]
        true_vector  = batch_scores[:, token_true_id]
        false_vector = batch_scores[:, token_false_id]
        batch_scores = torch.stack([false_vector, true_vector], dim=1)
        batch_scores = torch.nn.functional.log_softmax(batch_scores, dim=1)
        return batch_scores[:, 1].exp().tolist()

    docs   = [doc for doc, _ in retrieved]
    pairs  = [format_input(query, doc.page_content[:400]) for doc in docs]
    inputs = process_inputs(pairs)
    scores = compute_logits(inputs)   # already 0-1

    combined = sorted(zip(docs, scores, scores), key=lambda x: x[1], reverse=True)

    for rank, (doc, raw_s, _) in enumerate(combined, start=1):
        _qwen_debug_rows.append({
            "query":             query,
            "expected_standard": expected,
            "rank":              rank,
            "standard_code":     doc.metadata.get("standard_code"),
            "is_correct":        doc.metadata.get("standard_code") == expected,
            "score":             round(raw_s, 4),
            "is_relevant":       raw_s >= 0.5,
            "chunk_preview":     doc.page_content[:500],
        })

    return list(combined)

# To load qwen
# qwen_tokenizer, qwen_model, qwen_true_id, qwen_false_id, qwen_max_len, qwen_prefix_tokens, qwen_suffix_tokens = load_qwen()
# To not load qwen:
qwen_tokenizer = qwen_model = qwen_true_id = qwen_false_id = qwen_max_len = qwen_prefix_tokens = qwen_suffix_tokens = None
print("Qwen model loaded.")

Qwen model loaded.


## 8. Metrics
> Purely rank-based — no model scores involved. Fair and consistent across all rerankers.

In [15]:
def hit_at_k(docs_with_scores: list, expected: str, k: int) -> bool:
    """True if the correct standard appears anywhere in the top k results."""
    codes = [d.metadata.get("standard_code") for d, _ in docs_with_scores]
    return expected in codes[:k]


def rank_of(docs_with_scores: list, expected: str):
    """1-based position of the first correct chunk. None if not retrieved."""
    for i, (doc, _) in enumerate(docs_with_scores):
        if doc.metadata.get("standard_code") == expected:
            return i + 1
    return None


def mrr(rank) -> float:
    """Mean Reciprocal Rank. rank 1→1.0, rank 2→0.5, None→0.0"""
    return 1 / rank if rank else 0.0


def ndcg_at_k(docs_with_scores: list, expected: str, k: int) -> float:
    """Normalised Discounted Cumulative Gain at K."""
    import math
    codes = [d.metadata.get("standard_code") for d, _ in docs_with_scores[:k]]
    dcg   = 0.0
    for i, code in enumerate(codes):
        if code == expected:
            dcg += 1.0 / math.log2(i + 2)
            break
    idcg = 1.0 / math.log2(2)
    return round(dcg / idcg, 4) if idcg > 0 else 0.0

## 9. Evaluation

In [16]:
_qwen_debug_rows = []


def add_aggregate_column(rows: list) -> list:
    """
    Adds norm_score_aggregate to each row.
    At rank R: norm_score_aggregate = cumulative mean of norm_score for ranks 1..R.

    rank 1 agg = norm_score at rank 1
    rank 2 agg = mean(rank1, rank2)
    rank N agg = mean(rank1 ... rankN)

    A falling aggregate = top chunks are most relevant (reranker working correctly).
    A rising aggregate  = better chunks are sitting lower in the ranking.
    """
    query_groups = defaultdict(list)
    for row in rows:
        query_groups[row["query"]].append(row)

    result = []
    for query, group in query_groups.items():
        group_sorted = sorted(group, key=lambda r: r["rank"])
        running_sum  = 0.0
        for i, row in enumerate(group_sorted, start=1):
            running_sum += row["norm_score"]
            row["norm_score_aggregate"] = round(running_sum / i, 4)
        result.extend(group_sorted)
    return result


def run_evaluation_for_model(model_short_name, dbs, gold_standard,
                              cross_model, qwen_tokenizer, qwen_model,
                              qwen_true_id, qwen_false_id,
                              qwen_max_len, qwen_prefix_tokens, qwen_suffix_tokens):
    """
    Runs evaluation for one embedding model against all 3 rerankers.

    Returns:
        standard_rows — one row per query, rank-based metrics (Hit@K, MRR, NDCG)
        chunk_rows    — dict {baseline, cross, qwen}
                        each row has: raw_score, norm_score, norm_score_aggregate
    """
    standard_rows = []
    chunk_rows    = {"baseline": [], "cross": [], "qwen": []}

    for i, item in enumerate(gold_standard, start=1):
        query    = item["query"]
        expected = item["expected_standard"]

        print(f"  [{model_short_name}] Query {i}/{len(gold_standard)} "
              f"— Expected: {expected}")
        print(f"    {query[:70]}...")

        # ── Retrieve (FAISS) ──────────────────────────────────────────────
        retrieved = retrieve(query, dbs)

        # Normalise baseline L2 scores → 0-1
        base_raw_scores  = [score for _, score in retrieved]
        base_norm_scores = normalise_scores(base_raw_scores, "baseline")

        # ── CrossEncoder rerank ───────────────────────────────────────────
        cross_ranked = rerank_cross(query, retrieved, cross_model)

        # ── Qwen rerank ───────────────────────────────────────────────────
        if qwen_model is not None:
            qwen_ranked = rerank_qwen(
                query, retrieved[:QWEN_INPUT_K],
                qwen_tokenizer, qwen_model,
                qwen_true_id, qwen_false_id,
                qwen_max_len, qwen_prefix_tokens, qwen_suffix_tokens,
                expected=expected
            )
        else:
            qwen_ranked = []

        # ── Rebuild (doc, norm_score) lists for metric functions ──────────
        retrieved_norm = list(zip([doc for doc, _ in retrieved], base_norm_scores))
        cross_norm     = [(doc, norm) for doc, raw, norm in cross_ranked]
        qwen_norm      = [(doc, norm) for doc, raw, norm in qwen_ranked]

        b_rank = rank_of(retrieved_norm, expected)
        c_rank = rank_of(cross_norm,     expected)
        q_rank = rank_of(qwen_norm,      expected)

        standard_rows.append({
            "query":       query,
            "expected":    expected,
            # Baseline
            "base_rank":   b_rank,
            "base_hit1":   hit_at_k(retrieved_norm, expected, k=1),
            "base_hit3":   hit_at_k(retrieved_norm, expected, k=3),
            "base_hit5":   hit_at_k(retrieved_norm, expected, k=5),
            "base_mrr":    mrr(b_rank),
            "base_ndcg5":  ndcg_at_k(retrieved_norm, expected, k=5),
            # CrossEncoder
            "cross_rank":  c_rank,
            "cross_hit1":  hit_at_k(cross_norm, expected, k=1),
            "cross_hit3":  hit_at_k(cross_norm, expected, k=3),
            "cross_hit5":  hit_at_k(cross_norm, expected, k=5),
            "cross_mrr":   mrr(c_rank),
            "cross_ndcg5": ndcg_at_k(cross_norm, expected, k=5),
            # Qwen
            "qwen_rank":   q_rank,
            "qwen_hit1":   hit_at_k(qwen_norm, expected, k=1),
            "qwen_hit3":   hit_at_k(qwen_norm, expected, k=3),
            "qwen_hit5":   hit_at_k(qwen_norm, expected, k=5),
            "qwen_mrr":    mrr(q_rank),
            "qwen_ndcg5":  ndcg_at_k(qwen_norm, expected, k=5),
        })

        # ── Chunk rows — baseline ─────────────────────────────────────────
        for rank, ((doc, raw), norm) in enumerate(
                zip(retrieved, base_norm_scores), start=1):
            chunk_rows["baseline"].append({
                "query":              query,
                "expected_standard":  expected,
                "rank":               rank,
                "standard_retrieved": doc.metadata.get("standard_code"),
                "is_correct":         doc.metadata.get("standard_code") == expected,
                "raw_score":          round(float(raw), 4),
                "norm_score":         norm,
                "score_note":         "norm: 1=most similar, 0=least (inverted+scaled L2)",
                "chunk_text":         doc.page_content,
            })

        # ── Chunk rows — cross ────────────────────────────────────────────
        for rank, (doc, raw, norm) in enumerate(cross_ranked, start=1):
            chunk_rows["cross"].append({
                "query":              query,
                "expected_standard":  expected,
                "rank":               rank,
                "standard_retrieved": doc.metadata.get("standard_code"),
                "is_correct":         doc.metadata.get("standard_code") == expected,
                "raw_score":          round(float(raw), 4),
                "norm_score":         norm,
                "score_note":         "norm: sigmoid(logit) → 0-1 probability",
                "chunk_text":         doc.page_content,
            })

        # ── Chunk rows — qwen ─────────────────────────────────────────────
        for rank, (doc, raw, norm) in enumerate(qwen_ranked, start=1):
            chunk_rows["qwen"].append({
                "query":              query,
                "expected_standard":  expected,
                "rank":               rank,
                "standard_retrieved": doc.metadata.get("standard_code"),
                "is_correct":         doc.metadata.get("standard_code") == expected,
                "raw_score":          round(float(raw), 4),
                "norm_score":         norm,
                "score_note":         "norm: log_softmax prob, already 0-1",
                "chunk_text":         doc.page_content,
            })

    # Add aggregate column to all three reranker lists
    for key in chunk_rows:
        chunk_rows[key] = add_aggregate_column(chunk_rows[key])

    return standard_rows, chunk_rows


def print_model_summary(model_short_name, standard_rows):
    df = pd.DataFrame(standard_rows)
    n  = len(df)
    print(f"\n  {'─'*60}")
    print(f"  SUMMARY: {model_short_name}   (n={n} queries)")
    print(f"  {'─'*60}")
    print(f"  {'MODE':<10} {'Hit@1':<12} {'Hit@3':<12} {'Hit@5':<12} {'MRR':<10} {'NDCG@5'}")
    print(f"  {'─'*60}")
    for mode in ["base", "cross", "qwen"]:
        h1   = df[f"{mode}_hit1"].sum()
        h3   = df[f"{mode}_hit3"].sum()
        h5   = df[f"{mode}_hit5"].sum()
        mmrr = df[f"{mode}_mrr"].mean()
        ndcg = df[f"{mode}_ndcg5"].mean()
        print(
            f"  {mode.upper():<10}"
            f"{h1}/{n} ({h1/n*100:.0f}%){'':<4}"
            f"{h3}/{n} ({h3/n*100:.0f}%){'':<4}"
            f"{h5}/{n} ({h5/n*100:.0f}%){'':<4}"
            f"{mmrr:.3f}{'':<5}"
            f"{ndcg:.3f}"
        )
    print(f"  {'─'*60}")

## 10a. Run Evaluation
> Must run before 10b or 10c. Stores results in `all_model_results`.

In [17]:
all_model_results   = {}   # {model_short_name: (standard_rows, chunk_rows)}
all_model_summaries = []
_qwen_debug_rows    = []

for model_short_name, model_string in EMBEDDING_MODELS.items():

    print(f"\n{'='*60}")
    print(f"  EMBEDDING MODEL: {model_short_name}")
    print(f"  {model_string}")
    print(f"{'='*60}\n")

    embeddings = HuggingFaceEmbeddings(
        model_name=model_string,
        encode_kwargs={"normalize_embeddings": True},
    )

    model_dir = os.path.join(DB_SAVE_DIR, model_short_name)
    if not os.path.exists(model_dir):
        print(f"  ERROR: No saved DBs at {model_dir}. Run Section 5 first.")
        continue

    dbs = {}
    for chapter in chapter_chunks.keys():
        chapter_path = os.path.join(model_dir, chapter)
        if os.path.exists(chapter_path):
            dbs[chapter] = FAISS.load_local(
                chapter_path, embeddings,
                allow_dangerous_deserialization=True
            )
    print(f"[{model_short_name}] Loaded {len(dbs)} chapter databases.\n")

    start = time.time()
    standard_rows, chunk_rows = run_evaluation_for_model(
        model_short_name, dbs, gold_standard,
        cross_model, qwen_tokenizer, qwen_model,
        qwen_true_id, qwen_false_id,
        qwen_max_len, qwen_prefix_tokens, qwen_suffix_tokens,
    )
    elapsed = round(time.time() - start, 1)
    print(f"\n[{model_short_name}] Evaluation complete in {elapsed}s")

    print_model_summary(model_short_name, standard_rows)

    all_model_results[model_short_name] = (standard_rows, chunk_rows)

    df = pd.DataFrame(standard_rows)
    n  = len(df)
    for mode in ["base", "cross", "qwen"]:
        all_model_summaries.append({
            "embedding_model": model_short_name,
            "reranker":        mode,
            "n_queries":       n,
            "hit1":            int(df[f"{mode}_hit1"].sum()),
            "hit3":            int(df[f"{mode}_hit3"].sum()),
            "hit5":            int(df[f"{mode}_hit5"].sum()),
            "hit1_pct":        round(df[f"{mode}_hit1"].mean() * 100, 1),
            "hit3_pct":        round(df[f"{mode}_hit3"].mean() * 100, 1),
            "hit5_pct":        round(df[f"{mode}_hit5"].mean() * 100, 1),
            "mrr":             round(df[f"{mode}_mrr"].mean(), 4),
            "ndcg5":           round(df[f"{mode}_ndcg5"].mean(), 4),
        })

    _qwen_debug_rows.clear()

print("\nSection 10a complete. Run 10b and/or 10c to export files.")


  EMBEDDING MODEL: MiniLM
  sentence-transformers/all-MiniLM-L6-v2



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1356.95it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[MiniLM] Loaded 17 chapter databases.

  [MiniLM] Query 1/25 — Expected: ACC.1
    What screening criteria should be used when a patient arrives at the e...
  [MiniLM] Query 2/25 — Expected: ACC.1
    How should hospitals handle patients with communicable diseases during...
  [MiniLM] Query 3/25 — Expected: ACC.1.1
    How should hospitals inform patients about delays in diagnostic or tre...
  [MiniLM] Query 4/25 — Expected: ACC.2
    How should hospitals manage patient flow to prevent emergency departme...
  [MiniLM] Query 5/25 — Expected: ACC.2
    What are the requirements for managing patients waiting for inpatient ...
  [MiniLM] Query 6/25 — Expected: ACC.2.2
    What information must be provided to patients when they are admitted t...
  [MiniLM] Query 7/25 — Expected: ACC.2.3
    What criteria should be established for admission to intensive care un...
  [MiniLM] Query 8/25 — Expected: ACC.2.3
    How should hospitals determine when patients no longer need intensive ...
  [MiniLM

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1338.25it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BGE-small] Loaded 17 chapter databases.

  [BGE-small] Query 1/25 — Expected: ACC.1
    What screening criteria should be used when a patient arrives at the e...
  [BGE-small] Query 2/25 — Expected: ACC.1
    How should hospitals handle patients with communicable diseases during...
  [BGE-small] Query 3/25 — Expected: ACC.1.1
    How should hospitals inform patients about delays in diagnostic or tre...
  [BGE-small] Query 4/25 — Expected: ACC.2
    How should hospitals manage patient flow to prevent emergency departme...
  [BGE-small] Query 5/25 — Expected: ACC.2
    What are the requirements for managing patients waiting for inpatient ...
  [BGE-small] Query 6/25 — Expected: ACC.2.2
    What information must be provided to patients when they are admitted t...
  [BGE-small] Query 7/25 — Expected: ACC.2.3
    What criteria should be established for admission to intensive care un...
  [BGE-small] Query 8/25 — Expected: ACC.2.3
    How should hospitals determine when patients no longer n

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1259.42it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[BGE-large] Loaded 17 chapter databases.

  [BGE-large] Query 1/25 — Expected: ACC.1
    What screening criteria should be used when a patient arrives at the e...
  [BGE-large] Query 2/25 — Expected: ACC.1
    How should hospitals handle patients with communicable diseases during...
  [BGE-large] Query 3/25 — Expected: ACC.1.1
    How should hospitals inform patients about delays in diagnostic or tre...
  [BGE-large] Query 4/25 — Expected: ACC.2
    How should hospitals manage patient flow to prevent emergency departme...
  [BGE-large] Query 5/25 — Expected: ACC.2
    What are the requirements for managing patients waiting for inpatient ...
  [BGE-large] Query 6/25 — Expected: ACC.2.2
    What information must be provided to patients when they are admitted t...
  [BGE-large] Query 7/25 — Expected: ACC.2.3
    What criteria should be established for admission to intensive care un...
  [BGE-large] Query 8/25 — Expected: ACC.2.3
    How should hospitals determine when patients no longer n

## 10b. Export Standard-Level & Chunk-Level Files
> Can re-run independently after 10a.

Produces per embedding model:
- `STANDARD_EVAL_{model}.csv`
- `CHUNKS_{model}_{reranker}_top5.csv`
- `CHUNKS_{model}_{reranker}_top10.csv`

In [20]:
export_cols = [
    "query", "expected_standard", "rank",
    "standard_retrieved", "is_correct",
    "raw_score", "norm_score", "norm_score_aggregate",
    "score_note", "chunk_text",
]

for model_short_name, (standard_rows, chunk_rows) in all_model_results.items():

    # Standard-level file
    std_file = f"STANDARD_EVAL_{model_short_name}.csv"
    pd.DataFrame(standard_rows).to_csv(std_file, index=False)
    print(f"Saved: {std_file}")

    for reranker_name, rows in chunk_rows.items():
        if not rows:  # skip if empty (e.g. qwen was not run)
            continue
        df_all = (pd.DataFrame(rows)
                  .sort_values(["query", "rank"])
                  .reset_index(drop=True))

        # Top-10 (all chunks)
        top10_file = f"CHUNKS_{model_short_name}_{reranker_name}_top10.csv"
        df_all[export_cols].to_csv(top10_file, index=False)
        print(f"Saved: {top10_file}")

        # Top-5 (ranks 1-5 only)
        top5_file = f"CHUNKS_{model_short_name}_{reranker_name}_top5.csv"
        df_all[df_all["rank"] <= 5][export_cols].to_csv(top5_file, index=False)
        print(f"Saved: {top5_file}")

print("\nSection 10b complete.")

Saved: STANDARD_EVAL_MiniLM.csv
Saved: CHUNKS_MiniLM_baseline_top10.csv
Saved: CHUNKS_MiniLM_baseline_top5.csv
Saved: CHUNKS_MiniLM_cross_top10.csv
Saved: CHUNKS_MiniLM_cross_top5.csv
Saved: STANDARD_EVAL_BGE-small.csv
Saved: CHUNKS_BGE-small_baseline_top10.csv
Saved: CHUNKS_BGE-small_baseline_top5.csv
Saved: CHUNKS_BGE-small_cross_top10.csv
Saved: CHUNKS_BGE-small_cross_top5.csv
Saved: STANDARD_EVAL_BGE-large.csv
Saved: CHUNKS_BGE-large_baseline_top10.csv
Saved: CHUNKS_BGE-large_baseline_top5.csv
Saved: CHUNKS_BGE-large_cross_top10.csv
Saved: CHUNKS_BGE-large_cross_top5.csv

Section 10b complete.


## 10c. Export Chunk Relevance Summary Files
> Can re-run independently after 10a.

Produces:
- `CHUNK_RELEVANCE_SUMMARY_cross_model.csv` — one row per model combination
- `CHUNK_RELEVANCE_SUMMARY_per_query.csv` — one row per model × query
- `SUMMARY_ALL_MODELS.csv` — rank-based metrics summary

**Interpreting `agg_trend_5_to_10`:** negative = scores drop going deeper = reranker front-loads relevance (good). Positive = relevant chunks are ranked too low (bad).

In [22]:
per_query_rows   = []
cross_model_rows = []

for model_short_name, (standard_rows, chunk_rows) in all_model_results.items():
    for reranker_name, rows in chunk_rows.items():
        if not rows:
            continue
        df = pd.DataFrame(rows)

        for query, qgroup in df.groupby("query"):
            qgroup    = qgroup.sort_values("rank")
            agg_at_5  = qgroup[qgroup["rank"] == 5]["norm_score_aggregate"].values
            agg_at_10 = qgroup[qgroup["rank"] == 10]["norm_score_aggregate"].values

            per_query_rows.append({
                "embedding_model":     model_short_name,
                "reranker":            reranker_name,
                "query":               query,
                "expected_standard":   qgroup["expected_standard"].iloc[0],
                "agg_score_at_rank5":  round(float(agg_at_5[0]),  4) if len(agg_at_5)  else None,
                "agg_score_at_rank10": round(float(agg_at_10[0]), 4) if len(agg_at_10) else None,
                "n_correct_top5":      int(qgroup[qgroup["rank"] <= 5]["is_correct"].sum()),
                "n_correct_top10":     int(qgroup["is_correct"].sum()),
                "top1_is_correct":     bool(qgroup[qgroup["rank"] == 1]["is_correct"].values[0]),
            })

        per_q = pd.DataFrame([
            r for r in per_query_rows
            if r["embedding_model"] == model_short_name
            and r["reranker"] == reranker_name
        ])

        cross_model_rows.append({
            "embedding_model":       model_short_name,
            "reranker":              reranker_name,
            "mean_agg_score_rank5":  round(per_q["agg_score_at_rank5"].mean(),  4),
            "mean_agg_score_rank10": round(per_q["agg_score_at_rank10"].mean(), 4),
            "avg_correct_top5":      round(per_q["n_correct_top5"].mean(),  2),
            "avg_correct_top10":     round(per_q["n_correct_top10"].mean(), 2),
            "rank1_accuracy_pct":    round(per_q["top1_is_correct"].mean() * 100, 1),
            "agg_trend_5_to_10":     round(
                per_q["agg_score_at_rank10"].mean() -
                per_q["agg_score_at_rank5"].mean(), 4
            ),
        })

per_query_df   = pd.DataFrame(per_query_rows)
cross_model_df = pd.DataFrame(cross_model_rows)

# Print to console
print("\n" + "="*80)
print("  CHUNK RELEVANCE SUMMARY")
print("  agg_trend_5_to_10: negative = reranker front-loads relevance (good)")
print("="*80)
print(cross_model_df.to_string(index=False))

# Save
cross_model_df.to_csv("CHUNK_RELEVANCE_SUMMARY_cross_model.csv", index=False)
print("\nSaved: CHUNK_RELEVANCE_SUMMARY_cross_model.csv")

per_query_df.to_csv("CHUNK_RELEVANCE_SUMMARY_per_query.csv", index=False)
print("Saved: CHUNK_RELEVANCE_SUMMARY_per_query.csv")

pd.DataFrame(all_model_summaries).to_csv("SUMMARY_ALL_MODELS.csv", index=False)
print("Saved: SUMMARY_ALL_MODELS.csv")

print("\nSection 10c complete.")


  CHUNK RELEVANCE SUMMARY
  agg_trend_5_to_10: negative = reranker front-loads relevance (good)
embedding_model reranker  mean_agg_score_rank5  mean_agg_score_rank10  avg_correct_top5  avg_correct_top10  rank1_accuracy_pct  agg_trend_5_to_10
         MiniLM baseline                0.6270                 0.3779              2.52               3.00                72.0            -0.2491
         MiniLM    cross                0.8355                 0.5147              2.64               3.00                88.0            -0.3208
      BGE-small baseline                0.6237                 0.3673              2.60               3.28                64.0            -0.2563
      BGE-small    cross                0.8481                 0.5530              2.60               3.28                88.0            -0.2950
      BGE-large baseline                0.6210                 0.3768              2.68               3.40                92.0            -0.2443
      BGE-large    cross   